# Delta Lake no AWS Glue 6.0

Neste exemplo, que roda no runtime **AWS Glue 6.0**, seguimos a mesma sequência em **PySpark** e em **Spark SQL**: gravamos um CSV na `landing`, lemos do `s3`, criamos uma tabela **gerenciada** e uma **externa** no schema `bronze`, atualizamos a gerenciada com `MERGE INTO` e copiamos o resultado para a externa. Todas as células podem ser reexecutadas.

## 1. Sessão

O runtime **AWS Glue 6.0** já configura extensões, catálogos e acesso ao `s3`; só precisamos obter a sessão a partir do `GlueContext`.

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session

## 2. Preparar a origem

Gravamos duas levas de CSV (sem cabeçalho) em `landing/`: a carga inicial e as alterações. `overwrite` mantém a célula reexecutável.

In [ ]:
schema = "id INT, nome STRING, area STRING, ts BIGINT"

carga_inicial = spark.createDataFrame(
    [(1, "Ana", "Dados", 1), (2, "Bruno", "Plataforma", 1), (3, "Carla", "Segurança", 1)],
    schema,
)
alteracoes = spark.createDataFrame(
    [(2, "Bruno", "Dados", 2), (4, "Diego", "Plataforma", 2)],
    schema,
)

carga_inicial.write.mode("overwrite").csv("s3://notislake/landing/aws_delta_employees/")
alteracoes.write.mode("overwrite").csv("s3://notislake/landing/aws_delta_employees_updates/")

## 3. Ler do s3

Em PySpark informamos o schema ao ler o CSV.

In [ ]:
origem = spark.read.schema(schema).csv("s3://notislake/landing/aws_delta_employees/")
origem.orderBy("id").show()

Em Spark SQL lemos o mesmo caminho com `csv.` e convertemos as colunas (`_c0`, `_c1`, ...).

In [ ]:
SELECT
    CAST(_c0 AS INT) AS id,
    _c1 AS nome,
    _c2 AS area,
    CAST(_c3 AS BIGINT) AS ts
FROM csv.`s3://notislake/landing/aws_delta_employees/`
ORDER BY id

## 4. Criar o schema e as tabelas

Nenhum runtime cria schema sozinho: criar tabela em schema inexistente falha. Por isso começamos com `CREATE SCHEMA IF NOT EXISTS`, que também deixa a célula reexecutável; sem `LOCATION`, o schema vai para o bucket `notisbook-managed`.

Criamos a tabela Delta Lake de duas formas. Sem `LOCATION`, ela é **gerenciada**: os arquivos ficam em `s3://notisbook-managed/bronze/<tabela>` e o `DROP TABLE` apaga os arquivos. Com `LOCATION`, ela é **externa**: os arquivos ficam no bucket que escolhemos (aqui `notislake`) e o `DROP TABLE` os preserva. `IF NOT EXISTS` evita erro ao reexecutar.

In [ ]:
CREATE SCHEMA IF NOT EXISTS bronze

Tabela **gerenciada**, sem `LOCATION`:

In [ ]:
CREATE TABLE IF NOT EXISTS bronze.aws_delta_employees (
    id INT,
    nome STRING,
    area STRING,
    ts BIGINT
)
USING delta

Tabela **externa**, com `LOCATION` no bucket `notislake`:

In [ ]:
CREATE TABLE IF NOT EXISTS bronze.aws_delta_employees_ext (
    id INT,
    nome STRING,
    area STRING,
    ts BIGINT
)
USING delta
LOCATION 's3://notislake/bronze/aws_delta_employees'

## 5. Carregar e atualizar a tabela gerenciada com MERGE INTO

O `MERGE INTO` atualiza as linhas com o mesmo `id` e insere as novas, por isso reexecutar não duplica dados. Primeiro, em SQL, aplicamos a carga inicial a partir de uma view temporária lida do `s3`.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW carga_inicial_v AS
SELECT
    CAST(_c0 AS INT) AS id,
    _c1 AS nome,
    _c2 AS area,
    CAST(_c3 AS BIGINT) AS ts
FROM csv.`s3://notislake/landing/aws_delta_employees/`

In [ ]:
MERGE INTO bronze.aws_delta_employees AS t
USING carga_inicial_v AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

Agora, em PySpark, lemos as alterações do `s3`, registramos uma view temporária e executamos o `MERGE INTO` com `spark.sql`: o `id` 2 é atualizado e o `id` 4 é inserido.

In [ ]:
atualizacoes = spark.read.schema(schema).csv("s3://notislake/landing/aws_delta_employees_updates/")
atualizacoes.createOrReplaceTempView("alteracoes_v")

_ = spark.sql("""
    MERGE INTO bronze.aws_delta_employees AS t
    USING alteracoes_v AS s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

## 6. Copiar para a tabela externa

Usamos o mesmo `MERGE INTO` para levar o conteúdo da tabela gerenciada à externa; ela guarda os arquivos em `s3://notislake/bronze/aws_delta_employees`, que continuam lá mesmo depois de um `DROP TABLE`. Selecionamos só as quatro colunas de dados porque a origem pode ter colunas de controle.

In [ ]:
MERGE INTO bronze.aws_delta_employees_ext AS t
USING (SELECT id, nome, area, ts FROM bronze.aws_delta_employees) AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

## 7. Consultar o resultado

As duas tabelas terminam com 4 linhas, o mesmo a cada reexecução. Em PySpark:

In [ ]:
spark.table("bronze.aws_delta_employees").orderBy("id").show()
spark.table("bronze.aws_delta_employees_ext").orderBy("id").show()

E em Spark SQL, primeiro a gerenciada e depois a externa:

In [ ]:
SELECT id, nome, area, ts FROM bronze.aws_delta_employees ORDER BY id

In [ ]:
SELECT id, nome, area, ts FROM bronze.aws_delta_employees_ext ORDER BY id